# Above the Limit

In [ ]:
import os
import json
import pickle
import datetime
import networkx as nx
import pandas as pd
import numpy as np
from collections import defaultdict
from multiprocessing import Pool
import matplotlib.pyplot as plt
from tqdm import tqdm

from PyPDF2 import PdfMerger


In [ ]:
font_size = 24
scale_factor = 1.2
two_sided_font_size = font_size * scale_factor

plt.rcParams["font.size"] = font_size

ipv_color = {4: "tab:blue", 6: "tab:green"}


In [ ]:
REPO_ROOT = os.path.abspath("..")
fd = open(os.path.join(REPO_ROOT, "settings.json"))
parameters = json.load(fd)
for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
    if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
        parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))
fd.close()

working_dir = parameters["WORKING_DIR"]
data_dir = parameters["DATA_DIR"]
data_raw_dir = parameters["DATA_RAW_DIR"]
start_date = parameters["START_DATE"]
end_date = parameters["END_DATE"]
collectors = parameters["COLLECTORS"]
image_dir = parameters["IMAGE_DIR"]
tier1 = parameters["TIER1"]
tier1_asns = [item["asn"] for item in tier1]


In [ ]:
## Open stats output file (overwrites on every run)
numbers_dir = f"{data_dir}/processed/numbers"
os.makedirs(numbers_dir, exist_ok=True)
_stats = open(f"{numbers_dir}/11-Above_limit_instances.md", "w")
_stats.write("# Stats: 11-Above_limit_instances\n\n")
_stats.write(f"*Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
print("Stats file opened.")


## Load data

In [ ]:
start_date_obj = datetime.datetime.strptime(start_date, "%Y-%m-%d")
end_date_obj = datetime.datetime.strptime(end_date, "%Y-%m-%d")

delta = datetime.timedelta(hours=8)
all_times = []
current = start_date_obj
while current < end_date_obj:
    all_times.append(current)
    current += delta


### PeeringDB data

In [ ]:
filename = (
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
)
df_peeringdb = pd.read_pickle(filename)
df_peeringdb.head(2)


In [ ]:
filename = f"{data_dir}/processed/timeseries_prefix_announced_visibility.pkl"

with open(filename, "rb") as fd:
    announced_prefixes = pickle.load(fd)


### Number of Peers

In [ ]:
filename = f"{data_dir}/processed/peers/df_peers_ipv4.pkl"
with open(filename, "rb") as f:
    peers_ipv4 = pickle.load(f)

filename = f"{data_dir}/processed/peers/df_peers_ipv6.pkl"
with open(filename, "rb") as f:
    peers_ipv6 = pickle.load(f)


### Selected ASNs

In [ ]:
fd = open(f"{data_dir}/processed/selected_asns.pkl", "rb")
selected_asns = pickle.load(fd)
fd.close()

len(selected_asns)


### Filter Announced Prefixes and PeeringDB data to selected ASNs

In [ ]:
announced_prefixes = {
    asn: announced_prefixes[asn] for asn in selected_asns if asn in announced_prefixes
}

print(f'Number of AS with announced prefixes : {len(announced_prefixes)}')

df_peeringdb = df_peeringdb[df_peeringdb["asn"].isin(selected_asns)].copy()


## Find excedence events

In [ ]:
def compute_exceedence_event(asn, ipv, high_visibility="visibility_95"):
    """Return ALL clean limit crossings for (asn, ipv).

    A crossing = below the limit at t-2 and t-1, then above at t, with no
    temporal gap. We NO LONGER require a peer drop here (that filtering used to
    bake in the arbitrary 1% rule and the <5-peers guard). Every crossing is
    returned with its coincident ABSOLUTE peer drop, computed with a
    one-window-ahead horizon (enforcement can land the snapshot after the
    crossing). Whether a crossing is "impactful" is decided later, against the
    AS's own per-AS churn floor (see compute_churn_floors).
    """

    default_output = []

    # if there is no announced prefix for this ASN and IP version, we can ignore it
    if asn not in announced_prefixes:
        return default_output
    if ipv not in announced_prefixes[asn]:
        return default_output

    announced_prefixes_asn_ipv = announced_prefixes[asn][ipv][high_visibility]
    if len(announced_prefixes_asn_ipv) == 0:
        return default_output

    announced_prefixes_asn_ipv = {
        k: v for k, v in sorted(announced_prefixes_asn_ipv.items(), key=lambda x: x[0])
    }

    peers_ipv = peers_ipv4 if ipv == 4 else peers_ipv6
    peers_ipv_asn = peers_ipv[peers_ipv["asn"] == asn]
    if peers_ipv_asn.empty:
        print(f"ASN {asn} has no peer data for IP version {ipv}")
        return default_output

    peers_ipv_asn = peers_ipv_asn.iloc[0]
    peers_ipv_asn_dates = peers_ipv_asn["datetime"]
    peers_ipv_asn_num_peers = peers_ipv_asn["num_peers"]
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in zip(peers_ipv_asn_dates, peers_ipv_asn_num_peers)
    }
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in sorted(peers_ipv_asn.items(), key=lambda x: x[0])
    }

    # by default asn should exist in peeringdb since we are using selected_asns
    prefix_limits_asn = df_peeringdb[df_peeringdb["asn"] == asn].iloc[0]
    # however, it may be that there is no limit for this ASN and IP version
    prefix_limits_asn_ipv_date = prefix_limits_asn["dates"]
    prefix_limits_asn_ipv_count = prefix_limits_asn[f"limits_ipv{ipv}"]
    if prefix_limits_asn_ipv_count is None or prefix_limits_asn_ipv_count == 0:
        return default_output

    prefix_limits_asn_ipv = {
        date: count
        for date, count in zip(prefix_limits_asn_ipv_date, prefix_limits_asn_ipv_count)
    }

    # population guardrails: below the limit sometimes, above sometimes, and
    # below for at least half the year, so it has a meaningful non-exceedance
    # baseline to build a per-AS churn floor from.
    count_below = 0
    for date in announced_prefixes_asn_ipv:
        if date in prefix_limits_asn_ipv:
            if announced_prefixes_asn_ipv[date] <= prefix_limits_asn_ipv[date]:
                count_below += 1

    # discard if announce is always below limit
    if count_below == 0:
        return default_output

    # discard if announce is always above limit
    if count_below == len(announced_prefixes_asn_ipv):
        return default_output

    # discard if announce is below limit for less than half of the time
    if count_below < len(announced_prefixes_asn_ipv) / 2:
        return default_output

    # kept only for the plotting helpers below
    large_number_peers = np.mean(list(peers_ipv_asn.values())) > 10

    announced_prefixes_asn_ipv_dates = list(announced_prefixes_asn_ipv.keys())

    excedence_events = []
    for index in range(2, len(announced_prefixes_asn_ipv) - 1):

        previous_2_date = announced_prefixes_asn_ipv_dates[index - 2]
        previous_date = announced_prefixes_asn_ipv_dates[index - 1]
        current_date = announced_prefixes_asn_ipv_dates[index]
        next_date = announced_prefixes_asn_ipv_dates[index + 1]

        # check that there is no temporal gap between the dates
        if not (
            next_date - delta == current_date
            and current_date - delta == previous_date
            and previous_date - delta == previous_2_date
        ):
            continue

        # structural crossing: below at t-2 and t-1, then above at t
        if not (
            announced_prefixes_asn_ipv[previous_2_date]
            <= prefix_limits_asn_ipv[previous_2_date]
            and announced_prefixes_asn_ipv[previous_date]
            <= prefix_limits_asn_ipv[previous_date]
            and announced_prefixes_asn_ipv[current_date]
            > prefix_limits_asn_ipv[current_date]
        ):
            continue

        n_previous_peers = peers_ipv_asn[previous_date]
        n_current_peers = peers_ipv_asn[current_date]
        n_next_peers = peers_ipv_asn[next_date]

        n_previous_prefixes = announced_prefixes_asn_ipv[previous_date]
        n_current_prefixes = announced_prefixes_asn_ipv[current_date]

        # coincident ABSOLUTE peer drop, one-window-ahead horizon.
        # positive = peers lost. deepest peer count over {t, t+1}.
        deepest = min(n_current_peers, n_next_peers)
        abs_drop = n_previous_peers - deepest
        # is the deeper drop reflected at t+1 rather than t? drives drop_date in
        # the detailed analysis (matches the old check_next semantics)
        check_next = n_next_peers < n_current_peers
        percentage_drop = (
            abs_drop / n_previous_peers * 100 if n_previous_peers > 0 else 0.0
        )

        excedence_event = {
            "asn": asn,
            "ipv": ipv,
            "date": current_date,
            "peeringdb_limit_previous_date": prefix_limits_asn_ipv[previous_date],
            "peeringdb_limit_date": prefix_limits_asn_ipv[current_date],
            "n_prefixes_previous_date": n_previous_prefixes,
            "n_prefixes_date": n_current_prefixes,
            "n_previous_peers": n_previous_peers,
            "n_current_peers": n_current_peers,
            "n_next_peers": n_next_peers,
            "check_next": check_next,
            "abs_drop": abs_drop,
            "percentage_drop": percentage_drop,
            "is_visually_interesting": bool(
                percentage_drop > 10 and large_number_peers
            ),
        }
        excedence_events.append(excedence_event)

    return excedence_events


In [ ]:
# Per-AS "normal churn" baseline (replaces the arbitrary 1% peer-drop rule).
#
# For every AS that crosses its limit, we measure its typical peer loss from the
# windows in which it operates BELOW its limit (non-exceedance), and read off the
# 90/95/99th percentiles. A crossing is later called "impactful" only if its
# coincident peer drop exceeds that AS's OWN 95th-pct floor -- the bar is per-AS
# (size-aware) and data-driven, not a fixed 1%.
#
# Metric: ABSOLUTE peer drop, one-window-ahead horizon, computed identically to
# the crossings. For the control we only look ahead INTO another below-limit
# window, so a crossing can never leak into its own baseline (no arbitrary
# buffer needed -- the baseline lives entirely below the limit).

CHURN_PERCENTILES = (90, 95, 99)
MIN_CONTROL_WINDOWS = 20  # need >= this many non-exceedance transitions for a floor


def compute_churn_floors(asn, ipv, high_visibility="visibility_95"):
    """Return (floors_abs, control_abs, control_rel) or None if too few windows.

    floors_abs  : {90: x, 95: y, 99: z} absolute-peer-drop percentiles
    control_abs : list of absolute drops over below-limit windows
    control_rel : same drops as a % of previous peers (for the pooled CDF)
    """
    if asn not in announced_prefixes or ipv not in announced_prefixes[asn]:
        return None
    announced = announced_prefixes[asn][ipv][high_visibility]
    if len(announced) == 0:
        return None
    announced = {k: v for k, v in sorted(announced.items(), key=lambda x: x[0])}

    peers_ipv = peers_ipv4 if ipv == 4 else peers_ipv6
    peers_ipv_asn = peers_ipv[peers_ipv["asn"] == asn]
    if peers_ipv_asn.empty:
        return None
    peers_ipv_asn = peers_ipv_asn.iloc[0]
    peers = {
        date: num_peers
        for date, num_peers in zip(
            peers_ipv_asn["datetime"], peers_ipv_asn["num_peers"]
        )
    }

    prefix_limits_asn = df_peeringdb[df_peeringdb["asn"] == asn].iloc[0]
    limits = prefix_limits_asn[f"limits_ipv{ipv}"]
    if limits is None:
        return None
    limits = {date: c for date, c in zip(prefix_limits_asn["dates"], limits)}

    dates = list(announced.keys())

    def below(d):
        return d in limits and announced[d] <= limits[d]

    control_abs = []
    control_rel = []
    for i in range(1, len(dates)):
        prev_d, cur_d = dates[i - 1], dates[i]
        # consecutive, both endpoints below the limit (non-exceedance churn)
        if cur_d - delta != prev_d:
            continue
        if not (below(prev_d) and below(cur_d)):
            continue
        if prev_d not in peers or cur_d not in peers:
            continue
        pb = peers[prev_d]
        if pb == 0:
            continue

        # one-window-ahead horizon, but only INTO another below-limit window
        cand = [peers[cur_d]]
        if i + 1 < len(dates):
            nxt_d = dates[i + 1]
            if nxt_d - delta == cur_d and below(nxt_d) and nxt_d in peers:
                cand.append(peers[nxt_d])
        drop = pb - min(cand)

        control_abs.append(drop)
        control_rel.append(drop / pb * 100)

    if len(control_abs) < MIN_CONTROL_WINDOWS:
        return None

    floors_abs = {q: float(np.percentile(control_abs, q)) for q in CHURN_PERCENTILES}
    return floors_abs, control_abs, control_rel


In [ ]:
# Classify every crossing against its AS's own churn floor.
# impactful@95 (a real peer loss exceeding the AS's 95th-pct non-exceedance
# churn) is THE definition; 90/99 are kept for the sensitivity appendix.

excedence_events = {4: [], 6: []}                    # impactful @ 95th (feeds downstream)
crossings_all = {4: [], 6: []}                        # every crossing (sensitivity)
excedence_events_visually_interesting = {4: [], 6: []}

percentage_drops_all = {4: [], 6: []}                 # impactful@95 relative drops (downstream stats)
prefix_diff_all = {4: [], 6: []}
prefix_growth_all = {4: [], 6: []}

# sensitivity: impactful counts under each per-AS percentile floor
impactful_counts = {ipv: {q: 0 for q in CHURN_PERCENTILES} for ipv in [4, 6]}
# validation: pooled treatment (crossings) vs control (churn), relative % for CDF
treat_rel_all = {4: [], 6: []}
control_rel_all = {4: [], 6: []}
# enrichment: crossings clearing their own 95th floor vs the 5% baseline
n_crossings_total = {4: 0, 6: 0}
n_above_own95 = {4: 0, 6: 0}
n_as_no_floor = {4: 0, 6: 0}   # ASes dropped: < MIN_CONTROL_WINDOWS control windows

for ipv in [4, 6]:

    for asn in tqdm(selected_asns):
        crossings = compute_exceedence_event(asn, ipv, high_visibility="visibility_95")
        if len(crossings) == 0:
            continue

        floors_res = compute_churn_floors(asn, ipv, high_visibility="visibility_95")
        if floors_res is None:
            # no trustworthy per-AS baseline -> exclude from the impactful set
            n_as_no_floor[ipv] += 1
            continue
        floors_abs, _control_abs, control_rel = floors_res
        control_rel_all[ipv].extend(control_rel)

        for excedence_event in crossings:
            abs_drop = excedence_event["abs_drop"]

            # attach this AS's own floors to the event
            excedence_event["churn_floor_90"] = floors_abs[90]
            excedence_event["churn_floor_95"] = floors_abs[95]
            excedence_event["churn_floor_99"] = floors_abs[99]

            # impactful = a real loss (>0) that exceeds the AS's own floor
            for q in CHURN_PERCENTILES:
                excedence_event[f"impactful_{q}"] = bool(
                    abs_drop > 0 and abs_drop > floors_abs[q]
                )

            crossings_all[ipv].append(excedence_event)
            treat_rel_all[ipv].append(excedence_event["percentage_drop"])
            n_crossings_total[ipv] += 1
            if excedence_event["impactful_95"]:
                n_above_own95[ipv] += 1
            for q in CHURN_PERCENTILES:
                if excedence_event[f"impactful_{q}"]:
                    impactful_counts[ipv][q] += 1

            # the impactful@95 set drives everything downstream
            if excedence_event["impactful_95"]:
                excedence_events[ipv].append(excedence_event)

                if excedence_event["is_visually_interesting"]:
                    excedence_events_visually_interesting[ipv].append(excedence_event)

                percentage_drops_all[ipv].append(excedence_event["percentage_drop"])

                n_prev = excedence_event["n_prefixes_previous_date"]
                n_curr = excedence_event["n_prefixes_date"]
                prefix_diff_all[ipv].append(n_curr - n_prev)
                if n_prev > 0:
                    prefix_growth_all[ipv].append((n_curr - n_prev) / n_prev * 100)

for ipv in [4, 6]:
    n_cross = n_crossings_total[ipv]
    enr = n_above_own95[ipv] / n_cross * 100 if n_cross else 0
    print(
        f"IPv{ipv}: crossings={n_cross} | "
        f"impactful@95={impactful_counts[ipv][95]} "
        f"(@90={impactful_counts[ipv][90]}, @99={impactful_counts[ipv][99]}) | "
        f"ASes w/o floor (<{MIN_CONTROL_WINDOWS} ctrl)={n_as_no_floor[ipv]}"
    )
    print(
        f"   enrichment: {enr:.1f}% of crossings clear their own 95th floor "
        f"(vs the 5% baseline by construction)"
    )
    print(
        f"   visually interesting (impactful@95, >10% drop, >10 avg peers): "
        f"{len(excedence_events_visually_interesting[ipv])}"
    )


### Save all excedence events

In [ ]:
# impactful@95 set (feeds the detailed / critical analysis downstream)
events_filename = f"{data_dir}/processed/excedence_events.json"
with open(events_filename, "w") as fd:
    json.dump(excedence_events, fd, indent=4, default=str)

# every crossing, with its per-AS floors and impactful@90/95/99 flags, for the
# threshold-sensitivity appendix
crossings_filename = f"{data_dir}/processed/crossings_all.json"
with open(crossings_filename, "w") as fd:
    json.dump(crossings_all, fd, indent=4, default=str)

print(
    f"Saved impactful@95: IPv4={len(excedence_events[4])}, IPv6={len(excedence_events[6])} "
    f"| all crossings: IPv4={len(crossings_all[4])}, IPv6={len(crossings_all[6])}"
)


In [ ]:
_stats.write("## Impactful Exceedance Events (per-AS churn floor)\n\n")
_stats.write(
    "Impactful = coincident peer drop exceeds the AS's own 95th-pct "
    "non-exceedance churn floor (absolute peers, one-window horizon). "
    "The bar is per-AS and data-driven; no fixed 1%, no minimum-size cut.\n\n"
)

for ipv in [4, 6]:
    n_cross = n_crossings_total[ipv]
    n_imp = impactful_counts[ipv][95]
    base_rate = n_imp / n_cross * 100 if n_cross else 0
    n_interesting = len(excedence_events_visually_interesting[ipv])
    _stats.write(f"### IPv{ipv}\n\n")
    _stats.write(f"- Clean crossings analysed: {n_cross:,}\n")
    _stats.write(f"- Impactful events (> own 95th-pct floor): {n_imp:,}\n")
    _stats.write(
        f"- Base rate of impact: {n_imp:,}/{n_cross:,} = {base_rate:.1f}% of "
        f"crossings coincide with an above-baseline peer loss\n"
    )
    _stats.write(
        f"- ASes excluded (< {MIN_CONTROL_WINDOWS} control windows): "
        f"{n_as_no_floor[ipv]:,}\n"
    )
    _stats.write(
        f"- Visually interesting (> 10% drop + > 10 avg peers): {n_interesting:,}\n\n"
    )
_stats.flush()

# --- sensitivity appendix: impactful counts under 90 / 95 / 99 floors ---
_stats.write("## Threshold sensitivity: impactful under per-AS 90/95/99 floors\n\n")
_stats.write("| IP | Crossings | @90th | @95th | @99th |\n")
_stats.write("|----|-----------|-------|-------|-------|\n")
for ipv in [4, 6]:
    c = impactful_counts[ipv]
    _stats.write(
        f"| IPv{ipv} | {n_crossings_total[ipv]:,} | "
        f"{c[90]:,} | {c[95]:,} | {c[99]:,} |\n"
    )
_stats.write("\n")
_stats.flush()

# --- validation: drops at crossings vs ordinary churn (the reviewers' baseline) ---
# Two arguments, both independent of the base rate above:
#  (a) by construction, a flagged drop is in the top 5% of the AS's OWN churn;
#  (b) ordinary churn almost never produces a large drop, while crossings have a
#      much heavier tail -> see the CDF and the percentiles below.
_stats.write("## Churn-baseline validation (drops at crossings vs. ordinary churn)\n\n")
_stats.write(
    "Ordinary (non-exceedance) churn almost never produces a large peer drop, "
    "whereas exceedance crossings have a much heavier tail. This is the direct "
    "answer to the baseline question (38C/38D); see the CDF figure below.\n\n"
)
_stats.write("| IP | drop distribution | median | 95th | 99th | n |\n")
_stats.write("|----|-------------------|--------|------|------|---|\n")
for ipv in [4, 6]:
    t = np.asarray(treat_rel_all[ipv], float)
    c = np.asarray(control_rel_all[ipv], float)
    if len(t):
        _stats.write(
            f"| IPv{ipv} | crossings (treatment) | {np.median(t):.1f}% | "
            f"{np.percentile(t, 95):.1f}% | {np.percentile(t, 99):.1f}% | {len(t):,} |\n"
        )
    if len(c):
        _stats.write(
            f"| IPv{ipv} | ordinary churn (control) | {np.median(c):.1f}% | "
            f"{np.percentile(c, 95):.1f}% | {np.percentile(c, 99):.1f}% | {len(c):,} |\n"
        )
_stats.write("\n")
_stats.flush()


In [ ]:
_stats.write("## Prefix Growth at Crossing Events\n\n")
_stats.write("(Impactful@95 events; growth = prefix count at crossing vs. 8h prior)\n\n")
for ipv in [4, 6]:
    growth = prefix_growth_all[ipv]
    diff = prefix_diff_all[ipv]
    if len(growth) == 0:
        _stats.write(f"### IPv{ipv}\n\nNo data.\n\n")
        continue
    median_g = float(np.median(growth))
    mean_g = float(np.mean(growth))
    p25 = float(np.percentile(growth, 25))
    p75 = float(np.percentile(growth, 75))
    n_small = sum(1 for g in growth if g <= 10)
    n_large = sum(1 for g in growth if g > 50)

    median_diff = float(np.median(diff))
    mean_diff = float(np.mean(diff))
    p25_diff = float(np.percentile(diff, 25))
    p75_diff = float(np.percentile(diff, 75))


    _stats.write(f"### IPv{ipv}\n\n")
    _stats.write(f"- N events: {len(growth):,}\n")
    _stats.write(f"- Median growth: {median_g:.1f}%\n")
    _stats.write(f"- Mean growth: {mean_g:.1f}%\n")
    _stats.write(f"- IQR: [{p25:.1f}%, {p75:.1f}%]\n")
    _stats.write(f"- Events with <= 10% growth (gradual): {n_small:,} ({n_small/len(growth)*100:.1f}%)\n")
    _stats.write(f"- Events with > 50% growth (sudden spike): {n_large:,} ({n_large/len(growth)*100:.1f}%)\n\n")

    _stats.write(f"- Median prefix count difference: {median_diff:.1f}\n")
    _stats.write(f"- Mean prefix count difference: {mean_diff:.1f}\n")
    _stats.write(f"- IQR prefix count difference: [{p25_diff:.1f}, {p75_diff:.1f}]\n\n")

    print(f"IPv{ipv}: median={median_g:.1f}%, mean={mean_g:.1f}%, IQR=[{p25:.1f}%, {p75:.1f}%]")
    print(f"IPv{ipv}: median={median_diff:.1f}, mean={mean_diff:.1f}, IQR=[{p25_diff:.1f}, {p75_diff:.1f}]")
_stats.flush()


## Churn-baseline validation figure

Peer-drop CDF at exceedance crossings (treatment) vs. ordinary churn in
non-exceedance windows (control), pooled across ASes. If the crossing curve sits
to the right, drops at exceedances are not just ordinary churn, which is the
direct answer to the reviewers' baseline question (38C/38D). Shown in relative %
so ASes of different sizes are comparable; the impactful classification itself
uses the per-AS absolute floor.

In [ ]:
os.makedirs(f"{image_dir}/impact", exist_ok=True)

for ipv in [4, 6]:
    treat = np.sort(np.asarray(treat_rel_all[ipv], float))
    control = np.sort(np.asarray(control_rel_all[ipv], float))

    plt.figure(figsize=(8, 6))
    for name, arr, color in [
        ("Ordinary churn (control)", control, "tab:gray"),
        ("Exceedance crossings (treatment)", treat, "tab:red"),
    ]:
        if len(arr) == 0:
            continue
        y = np.arange(1, len(arr) + 1) / len(arr)
        plt.plot(arr, y, label=name, color=color, lw=2.5)

    plt.axvline(0, color="black", lw=0.8, ls=":")
    plt.xlim(-30, 100)
    plt.xlabel("Relative peer drop (%)  [positive = peers lost]", fontsize=two_sided_font_size)
    plt.ylabel("CDF", fontsize=two_sided_font_size)
    plt.grid(alpha=0.3)
    plt.legend(fontsize=font_size * 0.7, loc="lower right")

    plt.savefig(f"{image_dir}/impact/churn_baseline_cdf_ipv{ipv}.pdf", bbox_inches="tight", dpi=300)
    plt.savefig(f"{image_dir}/impact/churn_baseline_cdf_ipv{ipv}.png", bbox_inches="tight", dpi=300)
    plt.show()


## General excedance plots 

### Percentage prefix increase

In [ ]:
bins = np.logspace(0, 4, 50)

for ipv in [4, 6]:    
    plt.hist(prefix_growth_all[ipv], bins=bins, alpha=0.85, label=f"IPv{ipv}", color=ipv_color[ipv])
    # plt.yscale("log")
    plt.xscale("log")
    plt.xlabel("Prefix Growth (%)")
    plt.ylabel("Number of Events")
    plt.savefig(f"{image_dir}/above_limit/prefix_growth_hist_ipv{ipv}.pdf", bbox_inches="tight")
    plt.savefig(f"{image_dir}/above_limit/prefix_growth_hist_ipv{ipv}.png", bbox_inches="tight")
    plt.show()


### Prefix increase absolute values

In [ ]:
bins = np.logspace(0, 3.2, 51)
# bins = [int(b) for b in bins]
# bins = sorted(set(bins))


for ipv in [4, 6]:
    distribution = prefix_diff_all[ipv]

    plt.figure(figsize=(8, 6))

    plt.hist(prefix_diff_all[ipv], 
            bins=bins, 
            align="mid",
            alpha=0.85, 
            label=f"IPv{ipv}", color=ipv_color[ipv])
    
    median_increase = np.median(distribution)
    plt.axvline(
        median_increase,
        color="black",
        alpha=0.5,
        linestyle="--",
        linewidth=4,
    )
    # text
    plt.text(
        median_increase + 3,
        plt.ylim()[1] * 0.8,
        f"Median: {int(median_increase)}",
        color="black",
        alpha=0.7,
        fontsize=two_sided_font_size,
    )

    # plt.yscale("log")
    plt.ylabel("Number of Events", fontsize=two_sided_font_size)
    plt.xlabel("Prefix Count Difference", fontsize=two_sided_font_size)
    plt.grid(axis="y", alpha=0.3)
    plt.xscale("log")
    plt.xlim(bins[0], bins[-1])

    plt.savefig(f"{image_dir}/above_limit/prefix_diff_hist_ipv{ipv}.pdf", 
                bbox_inches="tight",
                dpi=300)
    plt.savefig(f"{image_dir}/above_limit/prefix_diff_hist_ipv{ipv}.png", 
                bbox_inches="tight",
                dpi=300)
    plt.show()


## Plot (visually interesting) excedence events

### A few events around the excedence time

In [ ]:
def plot_excedence_event(
    asn, ipv, excedence_event, high_visibility="visibility_95", save=False, show=True
):

    announced_prefixes_asn_ipv = announced_prefixes[asn][ipv][high_visibility]
    announced_prefixes_asn_ipv = {
        k: v for k, v in sorted(announced_prefixes_asn_ipv.items(), key=lambda x: x[0])
    }

    peers_ipv = peers_ipv4 if ipv == 4 else peers_ipv6
    peers_ipv_asn = peers_ipv[peers_ipv["asn"] == asn]
    peers_ipv_asn = peers_ipv_asn.iloc[0]
    peers_ipv_asn_dates = peers_ipv_asn["datetime"]
    peers_ipv_asn_num_peers = peers_ipv_asn["num_peers"]
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in zip(peers_ipv_asn_dates, peers_ipv_asn_num_peers)
    }
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in sorted(peers_ipv_asn.items(), key=lambda x: x[0])
    }

    # by default asn should exist in peeringdb since we are using selected_asns
    prefix_limits_asn = df_peeringdb[df_peeringdb["asn"] == asn].iloc[0]
    # however, it may be that there is no limit for this ASN and IP version, so we need to check if the limit is set for this ASN and IP version
    prefix_limits_asn_ipv_date = prefix_limits_asn["dates"]
    prefix_limits_asn_ipv_count = prefix_limits_asn[f"limits_ipv{ipv}"]

    prefix_limits_asn_ipv = {
        date: count
        for date, count in zip(prefix_limits_asn_ipv_date, prefix_limits_asn_ipv_count)
    }

    window_size = 20
    excedence_event_date = excedence_event["date"]
    index = all_times.index(
        excedence_event_date
    )  # find the index of the point in all_times

    left_bound = max(0, index - window_size)
    left_bound = all_times[left_bound]
    right_bound = min(len(all_times) - 1, index + window_size)
    right_bound = all_times[right_bound]

    plt.figure(figsize=(25, 6))

    plt.title(f"ASN {asn} - IPv{ipv}")

    plt.ylabel("Number of Prefixes / Limit")

    plt.step(
        announced_prefixes_asn_ipv.keys(),
        announced_prefixes_asn_ipv.values(),
        color="tab:green",
        lw=2,
        where="mid",
        label="Announced Prefixes",
    )

    plt.step(
        prefix_limits_asn_ipv.keys(),
        prefix_limits_asn_ipv.values(),
        where="mid",
        label="PeeringDB Prefix Limit",
        color="tab:orange",
    )

    plt.scatter(
        x=excedence_event_date,
        y=announced_prefixes_asn_ipv[excedence_event_date],
        color="tab:red",
        marker="X",
        s=100,
        label="Interesting Point",
        zorder=5,
    )

    plt.ylim(
        0,
        max(
            max(announced_prefixes_asn_ipv.values()),
            max(prefix_limits_asn_ipv.values()),
        )
        * 1.1,
    )

    plt.xlim(
        left_bound,
        right_bound,
    )

    plt.twinx()

    plt.ylabel("Number of Peers")
    plt.step(
        peers_ipv_asn.keys(),
        peers_ipv_asn.values(),
        where="mid",
        color="tab:purple",
        label="Number of Peers",
    )

    plt.xlim(
        left_bound,
        right_bound,
    )
    plt.grid(axis="both", alpha=0.3)

    if save:
        filename = f'{working_dir}/excedence_events_plots/{ipv}/{asn}_ipv{ipv}_{excedence_event_date.strftime("%Y-%m-%d_%H-%M-%S")}.pdf'
        plt.savefig(filename, bbox_inches="tight", dpi=300)
        plt.savefig(filename.replace(".pdf", ".png"), bbox_inches="tight", dpi=300)

    if show:
        plt.show()
    else:
        plt.close()


In [ ]:
n_plots = 5
for ipv in excedence_events_visually_interesting:
    print(f"IPv{ipv} - Top {n_plots} Visually Interesting Excedence Events:")
    for excedence_event in excedence_events_visually_interesting[ipv][:n_plots]:
        plot_excedence_event(excedence_event["asn"], ipv, excedence_event)


#### Save all plots to PDF

In [ ]:
for ipv in excedence_events_visually_interesting:
    for excedence_event in excedence_events_visually_interesting[ipv]:
        plot_excedence_event(
            excedence_event["asn"], ipv, excedence_event, save=True, show=False
        )


### Full year and save as PDF

In [ ]:
def plot_full_timeline_with_excedence_events(
    asn, ipv, excedence_events, high_visibility="visibility_95", save=False, show=True
):

    announced_prefixes_asn_ipv = announced_prefixes[asn][ipv][high_visibility]
    announced_prefixes_asn_ipv = {
        k: v for k, v in sorted(announced_prefixes_asn_ipv.items(), key=lambda x: x[0])
    }

    peers_ipv = peers_ipv4 if ipv == 4 else peers_ipv6
    peers_ipv_asn = peers_ipv[peers_ipv["asn"] == asn]
    peers_ipv_asn = peers_ipv_asn.iloc[0]
    peers_ipv_asn_dates = peers_ipv_asn["datetime"]
    peers_ipv_asn_num_peers = peers_ipv_asn["num_peers"]
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in zip(peers_ipv_asn_dates, peers_ipv_asn_num_peers)
    }
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in sorted(peers_ipv_asn.items(), key=lambda x: x[0])
    }

    # by default asn should exist in peeringdb since we are using selected_asns
    prefix_limits_asn = df_peeringdb[df_peeringdb["asn"] == asn].iloc[0]
    # however, it may be that there is no limit for this ASN and IP version, so we need to check if the limit is set for this ASN and IP version
    prefix_limits_asn_ipv_date = prefix_limits_asn["dates"]
    prefix_limits_asn_ipv_count = prefix_limits_asn[f"limits_ipv{ipv}"]

    prefix_limits_asn_ipv = {
        date: count
        for date, count in zip(prefix_limits_asn_ipv_date, prefix_limits_asn_ipv_count)
    }

    plt.figure(figsize=(25, 6))

    plt.title(f"ASN {asn} - IPv{ipv}")

    plt.ylabel("Number of Prefixes / Limit")

    plt.step(
        announced_prefixes_asn_ipv.keys(),
        announced_prefixes_asn_ipv.values(),
        color="tab:green",
        lw=2,
        where="mid",
        label="Announced Prefixes",
    )

    plt.step(
        prefix_limits_asn_ipv.keys(),
        prefix_limits_asn_ipv.values(),
        where="mid",
        label="PeeringDB Prefix Limit",
        color="tab:orange",
    )

    for excedence_event in excedence_events:
        excedence_event_date = excedence_event["date"]
        plt.scatter(
            x=excedence_event_date,
            y=announced_prefixes_asn_ipv[excedence_event_date],
            color="tab:red",
            marker="X",
            s=100,
            label="Interesting Point",
            zorder=5,
        )

    plt.ylim(
        0,
        max(
            max(announced_prefixes_asn_ipv.values()),
            max(prefix_limits_asn_ipv.values()),
        )
        * 1.1,
    )

    plt.xlim(
        all_times[0],
        all_times[-1],
    )

    plt.twinx()

    plt.ylabel("Number of Peers")
    plt.step(
        peers_ipv_asn.keys(),
        peers_ipv_asn.values(),
        where="mid",
        color="tab:purple",
        label="Number of Peers",
    )

    plt.xlim(
        all_times[0],
        all_times[-1],
    )
    plt.grid(axis="both", alpha=0.3)

    if save:
        filename = f"{working_dir}/excedence_events_plots/{ipv}/{asn}_ipv{ipv}_full_timeline.pdf"
        plt.savefig(filename, bbox_inches="tight", dpi=300)
        plt.savefig(filename.replace(".pdf", ".png"), bbox_inches="tight", dpi=300)

    if show:
        plt.show()
    else:
        plt.close()


In [ ]:
asn_ipv_with_excedence_events = []

for ipv in excedence_events_visually_interesting:
    for excedence_event in excedence_events_visually_interesting[ipv]:
        asn = excedence_event["asn"]
        asn_ipv_with_excedence_events.append((asn, ipv))

asn_ipv_with_excedence_events = set(asn_ipv_with_excedence_events)

for asn, ipv in asn_ipv_with_excedence_events:
    events_asn_ipv = []
    for excedence_event in excedence_events_visually_interesting[ipv]:
        if excedence_event["asn"] == asn:
            events_asn_ipv.append(excedence_event)
    plot_full_timeline_with_excedence_events(
        asn, ipv, events_asn_ipv, save=True, show=False
    )


## Stats

### Percentage peer drops

In [ ]:
for ipv in percentage_drops_all:
    median_drop = np.median(percentage_drops_all[ipv])
    mean_drop = np.mean(percentage_drops_all[ipv])

    print(f"IPv{ipv} - Median Percentage Drop: {median_drop:.2f}%")
    print(f"IPv{ipv} - Mean Percentage Drop: {mean_drop:.2f}%")


In [ ]:
_stats.write("## Peer Drop Distribution\n\n")
for ipv in [4, 6]:
    median_drop = float(__import__("numpy").median(percentage_drops_all[ipv]))
    mean_drop = float(__import__("numpy").mean(percentage_drops_all[ipv]))
    n_severe = sum(1 for d in percentage_drops_all[ipv] if d > 85)
    _stats.write(f"### IPv{ipv}\n\n")
    _stats.write(f"- Median peer drop: {median_drop:.2f}%\n")
    _stats.write(f"- Mean peer drop: {mean_drop:.2f}%\n")
    _stats.write(f"- ASes losing > 85% of peers: {n_severe:,}\n\n")
_stats.flush()


In [ ]:
bins = np.linspace(1, 100, 50)
for ipv in [4, 6]:
    color = ipv_color[ipv]

    plt.figure(figsize=(8, 6))
    plt.hist(percentage_drops_all[ipv], bins=bins, color=color, alpha=0.85)

    median_drop = np.median(percentage_drops_all[ipv])
    plt.axvline(
        median_drop,
        color="black",
        alpha=0.5,
        linestyle="--",
        linewidth=4,
    )
    # text
    plt.text(
        median_drop + 3,
        plt.ylim()[1] * 0.8,
        f"Median: {median_drop:.2f}%",
        color="black",
        alpha=0.7,
        fontsize=two_sided_font_size,
    )

    # plt.title(f"Distribution of percentage drops for IPv{ipv}")
    plt.xlabel("Relative Peer Loss (%)", fontsize=two_sided_font_size)
    plt.ylabel("Count", fontsize=two_sided_font_size)
    plt.grid(axis="y", alpha=0.3)
    plt.xlim(bins[0], bins[-1])

    plt.savefig(
        f"{image_dir}/impact/percentage_drop_histogram_ipv{ipv}.pdf",
        bbox_inches="tight",
        dpi=300,
    )
    plt.savefig(
        f"{image_dir}/impact/percentage_drop_histogram_ipv{ipv}.png",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()


## Detailed event analysis

We are going to check which peers were lost during the events and how important they are in terms of their AS rank. We will use the cone size as a proxy for the importance of the peer, as it indicates how many ASes are downstream of that peer.

### Load CAIDA AS Ranks

In [ ]:
df_as_rank = pd.read_json(f"{data_raw_dir}/AS_rank/asns.jsonl", lines=True)
df_as_rank.head(2)


In [ ]:
as_rank_dict = df_as_rank.set_index("asn")["rank"].to_dict()


def classify_rank(asn):
    if asn not in as_rank_dict:
        return -1, "Unknown"
    rank = as_rank_dict.get(asn)

    if asn in tier1_asns:
        return rank, "Tier-1"
    if rank <= 100:
        return rank, "Major"
    if rank <= 1000:
        return rank, "Regional"
    return rank, "Peripheral"


### Load cache version

#### Graphs & prefixes

Check if graphs can can be replace with peers

In [ ]:
# On-disk caches for the detailed analysis below.
#
# graphs_cache: FULL per-date peer graphs (all ASNs) -> ASN-agnostic, so it stays
#   valid even when the impactful set changes. Safe to reuse.
# prefixes_cache: per-date prefixes filtered to the CURRENT impactful ASN set
#   (selected_asns_excedence_events). It therefore goes STALE whenever that set
#   changes -- delete data/processed/cache/prefixes_cache.pkl to force a rebuild.
#   (Deleted for this camera-ready rerun: the per-AS churn floor changed the set.)

graphs_cache = {}
prefixes_cache = {}

graphs_cache_filename = f"{data_dir}/processed/cache/graphs_cache.pkl"
if os.path.exists(graphs_cache_filename):
    with open(graphs_cache_filename, "rb") as fd:
        graphs_cache = pickle.load(fd)

prefixes_cache_filename = f"{data_dir}/processed/cache/prefixes_cache.pkl"
if os.path.exists(prefixes_cache_filename):
    with open(prefixes_cache_filename, "rb") as fd:
        prefixes_cache = pickle.load(fd)


In [ ]:
overwrite_graph_cache = False

def load_graph(date):
    if date in graphs_cache:
        print(f"Graph for date {date} is in cache.")
        return graphs_cache[date]
    
    print(f"Graph for date {date} is not in cache. Loading from file...")

    # if we are here, it means that the graph for this date is not in the cache,
    # so we need to load it from the file and add it to the cache
    # trigger overwrite of the cache
    global overwrite_graph_cache
    overwrite_graph_cache = (
        True  # set to True to always overwrite the cache with the latest data
    )
    date_str = date.strftime("%Y%m%d.%H%M")
    filename = f"{data_dir}/processed/peers/graphs/peers_graph_{date_str}.pkl"
    fd = open(filename, "rb")
    graph = pickle.load(fd)
    fd.close()
    graphs_cache[date] = graph

    return graph


In [ ]:
# loading all prefixes for all ASN blow the memory, so we are going to load a single file and keep only the ASN that appears on the excedence_events

selected_asns_excedence_events = set()
for events in excedence_events:
    for event in excedence_events[events]:
        asn = event["asn"]
        selected_asns_excedence_events.add(asn)

overwrite_prefix_cache = False


def load_prefixes(date):
    if date in prefixes_cache:
        print(f"Prefixes for date {date} loaded from cache.")
        return prefixes_cache[date]

    print(f"Loading prefixes for date {date} from file...")

    # if we are here, it means that the prefixes for this date is not in the cache,
    # so we need to load it from the file and add it to the cache
    # trigger overwrite of the cache
    global overwrite_prefix_cache
    overwrite_prefix_cache = (
        True  # set to True to always overwrite the cache with the latest data
    )
    date_str = date.strftime("%Y%m%d_%H%M")
    filename = f"{data_dir}/processed/announced_visibility/visibility_{date_str}.pkl"
    fd = open(filename, "rb")
    prefixes = pickle.load(fd)
    fd.close()

    prefixes = prefixes["asn_data"]

    # filter by selected_asns
    prefixes_selected_asn = {
        asn: prefixes[asn]
        for asn in selected_asns_excedence_events
        if asn in prefixes
    }
    prefixes_cache[date] = prefixes_selected_asn

    del prefixes

    return prefixes_selected_asn


### Enchance excedence event: prefixes and peers

In [ ]:
delta = datetime.timedelta(hours=8)


excedence_events_detailed = {
    4: [],
    6: [],
}

for ipv in excedence_events:
    print(
        f"IPv{ipv} - Analyzing in detail all events..."
    )

    # for excedence_event in tqdm(excedence_events_visually_interesting[ipv]):
    for excedence_event in tqdm(excedence_events[ipv]):

        asn = excedence_event["asn"]

        date = excedence_event["date"]
        previous_date = date - delta
        # lets use drop data as it can be the current date or the next date depending on the case whe loading peers
        # but for prefixes date is correct
        drop_date = date
        check_next = excedence_event["check_next"]
        if check_next:
            drop_date = date + delta

        n_prefixes_previous_date = excedence_event["n_prefixes_previous_date"]
        n_prefixes_date = excedence_event["n_prefixes_date"]

        n_previous_peers = excedence_event["n_previous_peers"]
        n_current_peers = excedence_event["n_current_peers"]
        n_next_peers = excedence_event["n_next_peers"]

        # load prefixes for this ASN and IP version
        previous_date_prefixes = load_prefixes(previous_date)
        date_prefixes = load_prefixes(date)
        previous_date_prefixes_asn_ipv = previous_date_prefixes[asn][ipv]["prefixes"]
        date_prefixes_asn_ipv = date_prefixes[asn][ipv]["prefixes"]

        # keep those with high visibility
        previous_date_prefixes_asn_ipv = [
            prefix
            for prefix in previous_date_prefixes_asn_ipv
            if previous_date_prefixes_asn_ipv[prefix]["visibility"] >= 95
        ]
        date_prefixes_asn_ipv = [
            prefix
            for prefix in date_prefixes_asn_ipv
            if date_prefixes_asn_ipv[prefix]["visibility"] >= 95
        ]

        # sanity check: the number of prefixes in the graph should match the number of prefixes in the timeseries data
        assert len(previous_date_prefixes_asn_ipv) == n_prefixes_previous_date
        assert len(date_prefixes_asn_ipv) == n_prefixes_date

        # load peers graphs for this ASN and IP version
        previous_date_graph = load_graph(previous_date)
        drop_date_graph = load_graph(drop_date)

        previous_date_peers = previous_date_graph[f"G_ipv{ipv}"][asn]
        drop_date_peers = drop_date_graph[f"G_ipv{ipv}"][asn]

        # sanity check: the number of peers in the graph should match the number of peers in the timeseries data
        assert len(previous_date_peers) == n_previous_peers
        if check_next:
            assert len(drop_date_peers) == n_next_peers
        else:
            assert len(drop_date_peers) == n_current_peers

        lost_peers = set(previous_date_peers) - set(drop_date_peers)
        added_peers = set(drop_date_peers) - set(previous_date_peers)
        new_prefixes = set(date_prefixes_asn_ipv) - set(previous_date_prefixes_asn_ipv)
        new_prefixes = list(new_prefixes)

        lost_major_peers = []
        lost_all_peers = []
        for lost_peer in lost_peers:
            rank, lost_peer_rank = classify_rank(lost_peer)
            if lost_peer_rank in ["Tier-1", "Major"]:
                lost_major_peers.append((lost_peer, lost_peer_rank, rank))
            lost_all_peers.append((lost_peer, lost_peer_rank, rank))

        new_added_peers = []
        for added_peer in added_peers:
            rank, added_peer_rank = classify_rank(added_peer)
            new_added_peers.append((added_peer, added_peer_rank, rank))

        lost_major_peers = sorted(lost_major_peers, key=lambda x: x[2])
        lost_all_peers = sorted(lost_all_peers, key=lambda x: x[2])
        if len(new_added_peers) > 0:
            added_peers = sorted(new_added_peers, key=lambda x: x[2])
        else:
            added_peers = []

        # if len(lost_major_peers) > 0:
        detail_event = {
            "excedence_event": excedence_event,
            "lost_major_peers": lost_major_peers,
            "lost_all_peers": lost_all_peers,
            "added_peers": added_peers,
            "previous_prefixes": previous_date_prefixes_asn_ipv,
            "new_prefixes": new_prefixes,
            'is_critical': len(lost_major_peers) > 0
        }
        excedence_events_detailed[ipv].append(detail_event)


### Save cache

In [ ]:
# lets write the caches, if the file does not exist or if we set the overwrite flag to True
if not os.path.exists(graphs_cache_filename) or overwrite_graph_cache:
    with open(graphs_cache_filename, "wb") as fd:
        pickle.dump(graphs_cache, fd)

if not os.path.exists(prefixes_cache_filename) or overwrite_prefix_cache:
    with open(prefixes_cache_filename, "wb") as fd:
        pickle.dump(prefixes_cache, fd)


## Process the events and focus on the critical ones

where critical is that lost a tier-1 or top100 AS

In [ ]:
def get_min_rank(event):
    lost_peer = event["lost_all_peers"]
    return min(peer[2] for peer in lost_peer)


excedence_events_detailed = {
    ipv: sorted(excedence_events_detailed[ipv], key=get_min_rank) for ipv in excedence_events_detailed
}


### Save all detailed events

In [ ]:
events_filename = f"{data_dir}/processed/excedence_events_detailed.json"
with open(events_filename, "w") as fd:
    json.dump(excedence_events_detailed, fd, indent=4, default=str)


### Extract the critical events

In [ ]:
critical_events = {
    4: [],
    6: []
}

for ipv in excedence_events_detailed:
    for event in excedence_events_detailed[ipv]:
        if event['is_critical']:
            critical_events[ipv].append(event)


#### Save the critical detailed events

In [ ]:
critical_events_filename = f"{data_dir}/processed/critical_excedence_events.json"
with open(critical_events_filename, "w") as fd:
    json.dump(critical_events, fd, indent=4, default=str)


### Compute the critical event stats

In [ ]:
"""
Aggregate lost peers by AS Rank tier across all critical exceedance events
and print the LaTeX rows for Table tab:lost-peer-rank.

lost_peer entries: [asn, classification, rank]
classification: "Tier-1" | "Major" (rank ≤100) | "Regional" (rank ≤1000) | "Peripheral"
"""

classifications = ["Tier-1", "Major", "Regional", "Peripheral", "Unknown"]
results = {}
for ipv in [4, 6]:
    counts = {c: 0 for c in classifications}

    for event in critical_events[ipv]:
        for asn, classification, rank in event["lost_all_peers"]:
            counts[classification] += 1
    total = sum(counts.values())
    results[ipv] = {
        t: counts[t] / total * 100 if total > 0 else 0 for t in classifications
    }
    results[ipv]["total"] = total

r4, r6 = results[4], results[6]
print(f"% IPv4: {r4['total']} lost peers total | IPv6: {r6['total']} lost peers total")
for classification in classifications:
    print(
        f"{classification} & {r4[classification]:.1f}\\% & {r6[classification]:.1f}\\% \\\\"
    )


In [ ]:
_stats.write("## Lost Peer Rank Distribution\n\n")
_stats.write("| Tier | IPv4 lost (%) | IPv6 lost (%) |\n")
_stats.write("|------|--------------|--------------|\n")
classifications = ["Tier-1", "Major", "Regional", "Peripheral", "Unknown"]
for ipv in [4, 6]:
    counts = {c: 0 for c in classifications}
    for event in critical_events[ipv]:
        for asn, classification, rank in event["lost_all_peers"]:
            counts[classification] += 1
    total = sum(counts.values())
    results_rank = {
        t: counts[t] / total * 100 if total > 0 else 0 for t in classifications
    }
    results_rank["total"] = total
    if ipv == 4:
        r4 = results_rank
    else:
        r6 = results_rank
_stats.write(f"| (total peers lost) | {int(r4["total"]):,} | {int(r6["total"]):,} |\n")
for c in classifications:
    _stats.write(f"| {c} | {r4[c]:.1f}% | {r6[c]:.1f}% |\n")
_stats.write("\n")
_stats.flush()


In [ ]:
## Critical event counts and Tier-1-only breakdown
n_impactful = {ipv: len(excedence_events[ipv]) for ipv in [4, 6]}

_stats.write("## Critical Event Summary\n\n")
_stats.write("(Critical = lost at least one Tier-1 or Major peer)\n\n")
_stats.write("| IPV | Impactful | Critical | Critical % | Tier-1-only events | Tier-1-only % |\n")
_stats.write("|-----|-----------|----------|------------|-------------------|---------------|\n")

for ipv in [4, 6]:
    n_critical = len(critical_events[ipv])
    n_tier1 = sum(
        1 for event in critical_events[ipv]
        if any(classification == "Tier-1" for _, classification, _ in event["lost_all_peers"])
    )
    n_imp = n_impactful[ipv]
    pct_critical = n_critical / n_imp * 100 if n_imp > 0 else 0
    pct_tier1 = n_tier1 / n_imp * 100 if n_imp > 0 else 0
    _stats.write(
        f"| IPv{ipv} | {n_imp} | {n_critical} | {pct_critical:.1f}% | {n_tier1} | {pct_tier1:.1f}% |\n"
    )
    print(f"IPv{ipv}: {n_critical}/{n_imp} critical ({pct_critical:.1f}%), {n_tier1} lost at least one Tier-1 ({pct_tier1:.1f}%)")

_stats.write("\n")
_stats.flush()


In [ ]:
_stats.close()
print(f"Stats written to {numbers_dir}/11-Above_limit_instances.md")
